In [9]:
import pandas as pd
import numpy as np
from src import sp_eda as se
import openpyxl
import locale
import sys
sys.path.append('../')
pd.set_option('display.max_columns', None)

Insertamos el dataframe y lo copiamos

In [6]:
df_raw=pd.read_csv('../data/Acc_met.csv')

df=df_raw.copy()
df.sample(5)

,State,Date,acc_events,acc_sev_1,acc_sev_2,acc_sev_3,acc_sev_4,weather_events,typ_Cold,typ_Fog,typ_Hail,typ_Precipitation,typ_Rain,typ_Snow,typ_Storm,wea_sev_Heavy,wea_sev_Light,wea_sev_Moderate,wea_sev_Other,wea_sev_Severe,wea_sev_UNK,Name
11420,RI,2022-06-22,6,2,1,3,0,11.0,0.0,2.0,0.0,0.0,9.0,0.0,0.0,0.0,9.0,0.0,0.0,2.0,0.0,Rhode Island
6841,MO,2022-01-22,9,0,9,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Missouri
11489,RI,2022-09-05,8,1,5,2,0,27.0,0.0,0.0,0.0,1.0,26.0,0.0,0.0,6.0,13.0,7.0,0.0,0.0,1.0,Rhode Island
8016,NC,2022-09-03,106,0,93,6,7,61.0,2.0,28.0,0.0,0.0,31.0,0.0,0.0,0.0,30.0,1.0,0.0,30.0,0.0,North Carolina
6783,MN,2022-11-18,72,0,72,0,0,465.0,10.0,35.0,0.0,0.0,2.0,418.0,0.0,1.0,419.0,28.0,0.0,17.0,0.0,Minnesota


Movemos la columna de posición y cambiamos de nombre la columna con el codigo de estado.

In [7]:
col = "Name"
cols = list(df.columns)
cols.insert(1, cols.pop(cols.index(col)))
df= df[cols]
df = df.rename(columns={
    "State": "State_code",
    "Name": "State"
})



df.sample(5)

,State_code,State,Date,acc_events,acc_sev_1,acc_sev_2,acc_sev_3,acc_sev_4,weather_events,typ_Cold,typ_Fog,typ_Hail,typ_Precipitation,typ_Rain,typ_Snow,typ_Storm,wea_sev_Heavy,wea_sev_Light,wea_sev_Moderate,wea_sev_Other,wea_sev_Severe,wea_sev_UNK
11008,PA,Pennsylvania,2022-03-14,237,0,204,19,14,2.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
11407,RI,Rhode Island,2022-06-08,8,1,3,4,0,43.0,0.0,10.0,0.0,1.0,32.0,0.0,0.0,5.0,19.0,12.0,0.0,6.0,1.0
13143,VA,Virginia,2022-09-19,190,0,172,0,18,75.0,18.0,53.0,0.0,0.0,4.0,0.0,0.0,0.0,4.0,13.0,0.0,58.0,0.0
3607,IA,Iowa,2022-12-09,7,0,7,0,0,547.0,4.0,230.0,0.0,0.0,158.0,155.0,0.0,23.0,232.0,61.0,0.0,231.0,0.0
6819,MN,Minnesota,2022-12-29,75,0,75,0,0,184.0,3.0,129.0,0.0,0.0,16.0,36.0,0.0,0.0,52.0,13.0,0.0,119.0,0.0


In [10]:
se.eda_prelim(df)

----------
DIMENSIONES
El conjunto de datos presenta 14431 filas y 22 columnas
----------
INFORMACION DE COLUMNAS
<class 'pandas.DataFrame'>
RangeIndex: 14431 entries, 0 to 14430
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   State_code         14431 non-null  str    
 1   State              14431 non-null  str    
 2   Date               14431 non-null  str    
 3   acc_events         14431 non-null  int64  
 4   acc_sev_1          14431 non-null  int64  
 5   acc_sev_2          14431 non-null  int64  
 6   acc_sev_3          14431 non-null  int64  
 7   acc_sev_4          14431 non-null  int64  
 8   weather_events     13492 non-null  float64
 9   typ_Cold           13492 non-null  float64
 10  typ_Fog            13492 non-null  float64
 11  typ_Hail           13492 non-null  float64
 12  typ_Precipitation  13492 non-null  float64
 13  typ_Rain           13492 non-null  float64
 14  typ_Snow       

Se van a unificar el tipo de datos de las columnas y vamos a usar tipo int, ya que para contar eventos nos es más que suficiente y es más eficiente.
En las filas de severidad de eventos meteorologicos, vamos a sustituir NsN por 0.

In [12]:

cols = df.columns[7:21]

df[cols] = df[cols].fillna(0).astype("int64")


Por otro lado, en la clasificación de eventos meteorologicos, existen dos tipos que vamos a unificar, ya que la información que aportan por separado no aporta más que toda ella junta (other y unknown).


In [14]:
df["wea_sev_UNKNOWN"] = df["wea_sev_UNK"] + df["wea_sev_Other"]

df = df.drop(columns=["wea_sev_UNK", "wea_sev_Other"])

Ponemos los nombres de las columnas todas en mayúsculas por estandarizar

In [16]:
df.columns = df.columns.str.upper()


In [17]:
se.eda_prelim(df)

----------
DIMENSIONES
El conjunto de datos presenta 14431 filas y 21 columnas
----------
INFORMACION DE COLUMNAS
<class 'pandas.DataFrame'>
RangeIndex: 14431 entries, 0 to 14430
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   STATE_CODE         14431 non-null  str    
 1   STATE              14431 non-null  str    
 2   DATE               14431 non-null  str    
 3   ACC_EVENTS         14431 non-null  int64  
 4   ACC_SEV_1          14431 non-null  int64  
 5   ACC_SEV_2          14431 non-null  int64  
 6   ACC_SEV_3          14431 non-null  int64  
 7   ACC_SEV_4          14431 non-null  int64  
 8   WEATHER_EVENTS     14431 non-null  int64  
 9   TYP_COLD           14431 non-null  int64  
 10  TYP_FOG            14431 non-null  int64  
 11  TYP_HAIL           14431 non-null  int64  
 12  TYP_PRECIPITATION  14431 non-null  int64  
 13  TYP_RAIN           14431 non-null  int64  
 14  TYP_SNOW       